In [14]:
import re
import pandas as pd
import spacy
import ollama
from collections import defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [15]:
# import spacy.cli
# spacy.cli.download("en_core_web_sm")

In [16]:
# Load SpaCy NER model
nlp = spacy.load("en_core_web_sm")

In [17]:
# Configuration
OLLAMA_MODEL = "llama3.2:1B"
CATEGORIES = [
    "groceries", "fuel", "travel", "subscriptions", "dining", "health",
    "utilities", "shopping", "others"
]
ALERT_THRESHOLD = 10000  # ₹10,000

In [18]:
def query_ollama(prompt: str) -> str:
    response = ollama.chat(
        model=OLLAMA_MODEL,
        messages=[{"role": "user", "content": prompt}]
    )
    return response['message']['content'].strip()


In [19]:
def classify_transaction(description: str, amount: float) -> str:
    prompt = f"""
You are a financial assistant. Categorize the following bank transaction into one of the following categories:
{CATEGORIES}

If the transaction doesn’t clearly fit any category, return "others".

Transaction: "{description}"
Amount: ₹{amount}

Only return the category name.
"""
    return query_ollama(prompt)

In [20]:

def extract_merchant_from_description(desc: str) -> str:
    # Attempt to extract known UPI names or personal names or app identifiers
    parts = re.split(r'[\/\-]', desc.lower())
    candidates = [p for p in parts if re.search(r'(swiggy|zomato|amazon|flipkart|bharatpe|paytm|phonepe|airtel|vodafone|netflix|prime|tatasky|vi|ola|uber)', p)]
    if candidates:
        return candidates[0]
    # fallback: try to find a named person or label from SpaCy
    doc = nlp(desc)
    for ent in doc.ents:
        if ent.label_ in ["PERSON", "ORG"]:
            return ent.text
    return "unknown"

In [21]:
def standardize_merchants(df: pd.DataFrame) -> pd.DataFrame:
    raw_merchants = df["Description"].astype(str).apply(extract_merchant_from_description)
    unique = raw_merchants.unique().tolist()
    vectorizer = TfidfVectorizer().fit(unique)
    merchant_vectors = vectorizer.transform(unique)

    def closest_match(name):
        query_vec = vectorizer.transform([name])
        sims = cosine_similarity(query_vec, merchant_vectors).flatten()
        best_idx = sims.argmax()
        return unique[best_idx] if sims[best_idx] > 0.7 else name

    df["Standard_Merchant"] = raw_merchants.apply(closest_match)
    return df

In [22]:
def detect_recurring_ner(description: str) -> bool:
    doc = nlp(description)
    keywords = ["monthly", "subscription", "recurring", "every month", "auto-debit"]
    return any(ent.label_ in ["DATE", "TIME"] for ent in doc.ents) and any(k in description.lower() for k in keywords)

In [23]:
def process_finsentry_json(input_json: dict) -> dict:
    account_number = input_json.get("Account Number", "UNKNOWN")
    transactions = input_json.get("Transactions", [])

    df = pd.DataFrame(transactions)
    df["Debit"] = df["Debit"].fillna(0)
    df["Credit"] = df["Credit"].fillna(0)
    df["Description"] = df["Description"].fillna("")

    # Filter only debits (expenses)
    df = df[df["Debit"].astype(str) != ""]
    df["Amount"] = df["Debit"].astype(float)

    # Extract merchant names
    df = standardize_merchants(df)

    # Compose cleaner prompt description
    df["Text"] = "UPI transaction to " + df["Standard_Merchant"]

    categorized_summary = defaultdict(list)
    alerts = []

    for _, row in df.iterrows():
        merchant = row["Standard_Merchant"]
        raw_description = row["Description"]
        amount = float(row["Amount"])
        prompt_text = row["Text"]

        category = classify_transaction(prompt_text, amount)
        flags = []

        if amount > ALERT_THRESHOLD:
            alerts.append(f"High-value transaction at '{merchant}' – ₹{amount}")
            flags.append("HIGH_VALUE")

        if category == "others":
            alerts.append(f"Unclear category for transaction: '{prompt_text}'")
            flags.append("UNCLEAR_CATEGORY")

        if detect_recurring_ner(raw_description):
            alerts.append(f"Recurring transaction detected: '{prompt_text}'")
            flags.append("RECURRING")

        categorized_summary[category].append({
            "merchant": merchant,
            "amount": amount,
            "flags": flags
        })

    return {
        "account_number": account_number,
        "categorized_summary": categorized_summary,
        "alerts": alerts
    }

In [24]:
import json

def run_test_from_file(input_file_path: str, output_file_path: str):
    from pathlib import Path

    if not Path(input_file_path).exists():
        print(f"❌ Input file not found: {input_file_path}")
        return

    with open(input_file_path, "r") as f:
        input_data = json.load(f)

    result = process_finsentry_json(input_data)

    with open(output_file_path, "w") as f:
        json.dump(result, f, indent=2)

    print(f"✅ ML output written to: {output_file_path}")
    return output_file_path

In [25]:
run_test_from_file("input.json", "output.json")


/tmp/ipykernel_46968/2428914000.py:6: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["Debit"] = df["Debit"].fillna(0)


✅ ML output written to: output.json


'output.json'